# Equipo 2 — Circuito RC de membrana
**Proyecto GluA2-AMPA · Electrofisiología Molecular I · UdeG CUCEI 2026**

Simula el potencial postsináptico excitatorio (EPSP) en una membrana dendrítica modelada
como circuito RC, bajo corriente sináptica AMPAR en modos R y Q.

In [ ]:
import sys
sys.path.append('../shared')
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from parametros_compartidos import *

# ── V_rev del Equipo 1 ────────────────────────────────────────────────────
# ACTUALIZAR con los valores del Equipo 1 cuando estén disponibles
# Por ahora se usan valores provisionales para desarrollo
V_REV_R = None   # ← reemplazar con el valor del Equipo 1 (mV)
V_REV_Q = None   # ← reemplazar con el valor del Equipo 1 (mV)

if V_REV_R is None:
    print('AVISO: V_rev provisional hasta recibir del Equipo 1')
    # Estimado GHK provisional
    V_REV_R = 0.0
    V_REV_Q = 5.0

print(f'V_rev Modo R = {V_REV_R} mV')
print(f'V_rev Modo Q = {V_REV_Q} mV')

## 1. Parámetros del parche de membrana

In [ ]:
# Parche isopotencial de 1 cm²
area_cm2 = 1.0

Cm   = CM_UF_CM2 * area_cm2    # µF
Rm   = RM_KOHM_CM2 / area_cm2  # kΩ  (R = Rm_specific / area)
g_pas = 1.0 / Rm                # µS  (conductancia de fuga)
E_pas = V_REPOSO_MV             # mV

tau_m = Rm * Cm * 1e3           # ms  (kΩ * µF → ms)

print(f'Cm   = {Cm} µF')
print(f'Rm   = {Rm} kΩ')
print(f'g_pas = {g_pas:.4f} µS')
print(f'τ_m  = {tau_m:.1f} ms  (= R_m · C_m)')

## 2. Conductancia sináptica (doble exponencial)

In [ ]:
G_MAX_NS = 0.5e-3 * 15   # nS → µS: 0.5 nS × 15 sinapsis = 7.5 nS
tau_r = TAU_RISE_MS
tau_d = TAU_DECAY_MS

def g_syn(t, t_stim=0.0):
    """Conductancia sináptica AMPAR (doble exponencial), en µS."""
    dt = t - t_stim
    if dt < 0:
        return 0.0
    return G_MAX_NS * (np.exp(-dt / tau_d) - np.exp(-dt / tau_r))

# Verificar normalización
t_test = np.linspace(0, 20, 2000)
g_test = np.array([g_syn(t) for t in t_test])
print(f'g_syn pico ≈ {g_test.max()*1000:.4f} nS  (esperado < {G_MAX_NS*1000:.1f} nS por normalización doble exp)')

## 3. EDO del circuito RC — EPSP único

In [ ]:
def dydt_RC(t, y, V_rev, stim_times):
    """EDO: C_m * dV/dt = -g_pas*(V-E_pas) - g_syn(t)*(V-V_rev)"""
    V = y[0]
    g_total = sum(g_syn(t, ts) for ts in stim_times)
    dVdt = (-g_pas * (V - E_pas) - g_total * (V - V_rev)) / Cm
    return [dVdt]

# Simulación: EPSP único
t_span = (0, 50)    # ms
t_eval = np.linspace(*t_span, 5000)
y0 = [V_REPOSO_MV]

sol_R = solve_ivp(dydt_RC, t_span, y0, t_eval=t_eval,
                  args=(V_REV_R, [5.0]), method='RK45')
sol_Q = solve_ivp(dydt_RC, t_span, y0, t_eval=t_eval,
                  args=(V_REV_Q, [5.0]), method='RK45')

amp_R = sol_R.y[0].max() - V_REPOSO_MV
amp_Q = sol_Q.y[0].max() - V_REPOSO_MV
print(f'Amplitud EPSP Modo R: {amp_R:.3f} mV')
print(f'Amplitud EPSP Modo Q: {amp_Q:.3f} mV')

## 4. Tren de 10 estímulos a 50 Hz

In [ ]:
isi_ms = 1000.0 / 50   # 20 ms entre estímulos
stim_train = [5.0 + i * isi_ms for i in range(10)]

t_span_train = (0, 300)
t_eval_train = np.linspace(*t_span_train, 30000)

sol_R_train = solve_ivp(dydt_RC, t_span_train, y0, t_eval=t_eval_train,
                        args=(V_REV_R, stim_train), method='RK45')
sol_Q_train = solve_ivp(dydt_RC, t_span_train, y0, t_eval=t_eval_train,
                        args=(V_REV_Q, stim_train), method='RK45')

print(f'Tren completado: {len(stim_train)} estímulos a 50 Hz')

## 5. Figura 3 del manuscrito

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A — EPSP único
axes[0].plot(sol_R.t, sol_R.y[0], 'b-', lw=2, label=f'Modo R  (Δ={amp_R:.2f} mV)')
axes[0].plot(sol_Q.t, sol_Q.y[0], 'r-', lw=2, label=f'Modo Q  (Δ={amp_Q:.2f} mV)')
axes[0].set_xlim(0, 50)
axes[0].set_xlabel('Tiempo (ms)')
axes[0].set_ylabel('V_m (mV)')
axes[0].set_title('EPSP único')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel B — Tren
axes[1].plot(sol_R_train.t, sol_R_train.y[0], 'b-', lw=1.5, label='Modo R')
axes[1].plot(sol_Q_train.t, sol_Q_train.y[0], 'r-', lw=1.5, label='Modo Q')
for ts in stim_train:
    axes[1].axvline(ts, color='gray', lw=0.6, linestyle='--', alpha=0.5)
axes[1].set_xlabel('Tiempo (ms)')
axes[1].set_ylabel('V_m (mV)')
axes[1].set_title('Tren 10 EPSPs a 50 Hz')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Circuito RC de membrana — GluA2 Modo R vs Modo Q', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('figura3_EPSP.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figura guardada: figura3_EPSP.png')